# Strategy 14. Naive strategy with graph schema

Evaluating strategy 14 - naive approach with graph schema - on BioMix test-set. Using enhanced schema.


In [1]:
import os
from dotenv import load_dotenv
from tqdm import tqdm
import json
import pandas as pd

# Load environment variables from the .env file
load_dotenv()

# Retrieve the variables from the environment
neo4j_uri = os.getenv("NEO4J_URI")
neo4j_username = os.getenv("NEO4J_USERNAME")
neo4j_password = os.getenv("NEO4J_PASSWORD")

# Check if any variable is missing
if not all([neo4j_uri, neo4j_username, neo4j_password]):
    raise EnvironmentError("One or more environment variables are missing: NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD")

print(f"Accessing OpenTargets at {neo4j_uri} as user {neo4j_username}")



ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/haikel.bogale/Documents/RanchoBio/Analysis/Internal_Work/RIW_190_NL_to_KG_benchmarking/.venv/lib/python3.9/site-packages/traitlets/traitlets.py", line 632, in get
    value = obj._trait_values[self.name]
KeyError: '_control_lock'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/haikel.bogale/Documents/RanchoBio/Analysis/Internal_Work/RIW_190_NL_to_KG_benchmarking/.venv/lib/python3.9/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/Users/haikel.bogale/Documents/RanchoBio/Analysis/Internal_Work/RIW_190_NL_to_KG_benchmarking/.venv/lib/python3.9/site-packages/ipykernel/kernelbase.py", line 301, in dispatch_control
    async with self._control_lock:
  File "/Users/haikel.bogale/Documents/RanchoBio/Analysis/Internal_Work/RIW_190_NL_to_KG_benchmarking/.venv/lib/python3.9

Accessing OpenTargets at bolt+s://pistoia.neo4j.rbsapp.net:7687 as user neo4j


Wrappers for LLMs and KG

In [2]:
# Langchain wrappers for different models

import re
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic

chat_models = {}

def new_ChatModel(model):
    if re.search(r"^gpt", model):
        return ChatOpenAI(model = model)
    elif re.search(r"^o1", model):
        return ChatOpenAI(model = model, temperature = 1)
    elif re.search(r"^claude", model):
        return ChatAnthropic(model = model)
    else:
        raise ValueError(f"Unsupported model: {model}")

def ChatModel(model):
    if model in chat_models:
        return chat_models[model]
    else:
        chat_models[model] = new_ChatModel(model)
        return chat_models[model]


In [3]:
# Cypher data extraction

def extract_cypher(message):
    text = message
    pattern = r"```cypher(.*?)```"
    matches = re.findall(pattern, text, re.DOTALL)
    try:
        return [match.strip() for match in matches]
    except Exception:
        raise ValueError(f"Failed to parse: {message}")

  
from py2neo import Graph

graph = Graph(
    neo4j_uri,
    auth=(neo4j_username, neo4j_password)
)

In [4]:
# Some queries take a very long time to run. This will deal with timeout

import threading

class TimeoutThread(threading.Thread):
    def __init__(self, func, *args, **kwargs):
        threading.Thread.__init__(self)
        self.func = func
        self.args = args
        self.kwargs = kwargs
        self.result = None
        self.error = None

    def run(self):
        try:
            self.result = self.func(*self.args, **self.kwargs)
        except Exception as e:
            self.error = e

def run_with_timeout(func, timeout, *args, **kwargs):
    thread = TimeoutThread(func, *args, **kwargs)
    thread.start()
    thread.join(timeout)

    if thread.is_alive():
        raise TimeoutError("Function execution timed out")
    elif thread.error:
        raise thread.error
    return thread.result

In [5]:
# convenience functions for data retrieval from graph

import time

def query_cypher_graph(graph, query):
    return graph.query(query)

def query_graph(llm_output):
    try:
        cypher_query = extract_cypher(llm_output)
    except Exception as e:
        return [{
            "query":None,
            "success":False,
            "exception":str(e)
        }]

    cypher_results = []
    for query in cypher_query:
        try:
            start = time.time()
            result = run_with_timeout(query_cypher_graph, 20, graph, query)
            duration = time.time() - start
            cypher_results.append({
                "query":query,
                "success":True,
                "result": list(result),
                "time": duration
                })
        except Exception as e:
            cypher_results.append({
                "query":query,
                "success":False,
                "exception":str(e)
            })
    return cypher_results


def process_results(todo, llm_answers, cypher_results):
    results = []
    for t,llm,res in zip(todo, llm_answers, cypher_results):
        out = {
            "model" : t[0],
            "question" : t[1],
            "llm_answer" : llm,
            "cypher_output": res,
            "n_cypher_queries" : len(res)
        }
        if len(res) > 0:
            out.update({
                "query": res[0].get('query',''),
                "success": res[0]['success']
            })
            if res[0]['success']:
                out.update({
                    "results": res[0]['result'],
                    "time": res[0]['time'],
                    "count" : len(res[0]['result'])
                })
            else:
                out.update({
                    "error": res[0]['exception']
                })
        results.append(out)
    return results



## Questions

Loading from an extended biomix test-set

In [6]:
questions = pd.read_csv("../biomix/testset/biomix_true_false_selected_augmented.csv")

## Running template-based query

- 14b - enchanced schema


In [7]:
# Graph schema

from langchain_community.graphs import Neo4jGraph

graph = Neo4jGraph(url=neo4j_uri, username=neo4j_username, password=neo4j_password)
graph.refresh_schema()

normal_schema = graph.schema

enhanced_graph = Neo4jGraph(
    url=neo4j_uri,
    username=neo4j_username,
    password=neo4j_password,
    enhanced_schema=True,
)

enhanced_schema = enhanced_graph.schema

print(enhanced_schema)

/var/folders/xh/h0s_k2yj7vl5lcl4dyjlstxr0000gp/T/ipykernel_85247/506362744.py:5: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-neo4j package and should be used instead. To use it run `pip install -U :class:`~langchain-neo4j` and import as `from :class:`~langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(url=neo4j_uri, username=neo4j_username, password=neo4j_password)
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The procedure has a deprecated field. ('config' used by 'apoc.meta.graphSample' is deprecated.)} {position: line: 1, column: 1, offset: 0} for query: "CALL apoc.meta.graphSample() YIELD nodes, relationships RETURN nodes, [rel in relationships | {nam

Node properties:
- **Entity**
  - `literature`: LIST Min Size: 1, Max Size: 1
  - `score`: FLOAT Example: "0.03"
  - `source`: STRING Example: "Open Targets"
  - `licence`: STRING Example: "https://platform-docs.opentargets.org/licence"
  - `version`: STRING Example: "22.11"
  - `id`: STRING Example: "obi:1110122"
  - `preferred_id`: STRING Example: "obi"
  - `code`: STRING Example: "http://purl.obolibrary.org/obo/OBI_1110122"
  - `name`: STRING Example: "pathological process"
  - `targetInModel`: STRING 
  - `targetInModelMgiId`: STRING 
  - `targetFromSourceId`: STRING 
  - `approvedSymbol`: STRING 
  - `biotype`: STRING 
  - `modelPhenotypeLabel`: STRING 
- **ThingWithTaxon**
  - `code`: STRING Example: "http://purl.obolibrary.org/obo/OBI_1110122"
  - `name`: STRING Example: "pathological process"
  - `description`: STRING Example: "Abnormal, harmful processes caused by or associate"
  - `source`: STRING Example: "Open Targets"
  - `licence`: STRING Example: "https://platform-docs.o

In [8]:
from langchain_core.prompts import PromptTemplate

# The following was adapted to guide LLM models (mainly gpt5 and up) for producing correct and extractable cypher queries

system_prompt_generic = """
You are a biological data scientist with expertise in knowledge graphs and Neo4j databases.

Your task: Generate syntactically correct Cypher queries to answer scientific questions about genes, diseases, and drugs.

Guidelines:
1. Always wrap your Cypher query in markdown code blocks with the 'cypher' language identifier:
```cypher
[insert cypher statement]
```

2. Use labels() for nodes, not type() - type() is only for relationships
3. Use case-insensitive matching for names: toLower() or CONTAINS with (?i) regex
4. Handle name variations (e.g., "Polycythemia Vera" vs "polycythemia vera")
5. Return relevant properties: id, name, score, source, etc.
6. Use ORDER BY and LIMIT when appropriate to manage result size
7. Add WHERE clauses to filter results effectively
8. For multiple label checks, use: WHERE any(l IN labels(node) WHERE l IN ['Label1', 'Label2'])
   NOT: WHERE (node:Label1 OR node:Label2)
9. For NULL handling in ORDER BY, use: ORDER BY coalesce(property, 0) DESC
   NOT: ORDER BY property DESC NULLS LAST

Common patterns:
- Gene to Disease: MATCH (g:Gene)-[:IS_PART_OF]->(assoc:GeneToDiseaseAssociation)<-[:IS_PART_OF]-(d:Disease)
- Check for property: WHERE g.approvedSymbol = 'GENE_NAME'
- Case-insensitive search: WHERE toLower(d.name) CONTAINS 'disease name'

{schema}

Output only the Cypher query wrapped in ```cypher code blocks. Do not include explanations unless requested.
"""




normal_schema_description = f"""\
This is graph schema:
--------------------------------------------
{normal_schema}
--------------------------------------------    
"""

enhanced_schema_description = f"""\
This is graph schema:
--------------------------------------------
{enhanced_schema}
--------------------------------------------    
"""

system_prompt_normal_schema = system_prompt_generic.format(schema = normal_schema_description)
system_prompt_enhanced_schema = system_prompt_generic.format(schema = enhanced_schema_description)

template_query = """
{question}
"""

user_template = PromptTemplate.from_template(template_query)

# Option 14b - enhanced schema

In [9]:
from langchain.schema import AIMessage, HumanMessage, SystemMessage

# models = ["gpt-4o", "claude-3-5-sonnet-20240620", "open-mistral-7b", "o1-preview-2024-09-12"]
models = ["gpt-4o", "gpt-5", "gpt-5.2-2025-12-11", "claude-sonnet-4-20250514", "claude-sonnet-4-5-20250929"]
niter = 1
todo = [(m,r['text']) for _,r in questions.iterrows() for m in models for _ in range(niter)]

def run_llm_14b(llm_model, question):
    try:
        llm = ChatModel(model = llm_model)
        user_prompt = user_template.invoke({"question": question})

        messages = [
            SystemMessage(content=system_prompt_enhanced_schema),
            HumanMessage(content=user_prompt.text)
        ]
        result = llm.invoke(messages)
        return result.content
    except Exception as e:
        print(e)
        return None


llm_answers = []
for llm_model, question in tqdm(todo, desc="Prompting LLM"):
    llm_answers.append(run_llm_14b(llm_model, question))


Prompting LLM: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 500/500 [1:42:28<00:00, 12.30s/it]


In [10]:
import time
for i, (llm_model, question) in enumerate(todo):
    if not llm_answers[i]:
        llm_answers[i] = run_llm_14b(llm_model, question)
        time.sleep(2)

In [11]:
cypher_results = []
for answer in tqdm(llm_answers, desc="Querying graph"):
    cypher_results.append(query_graph(answer))

Querying graph:  59%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▋                                                                                                           | 296/500 [22:29<20:39,  6.08s/it]Transaction failed and will be retried in 1.0627577459861328s (The allocation of an extra 2.0 MiB would use more than the limit 21.0 GiB. Currently using 21.0 GiB. dbms.memory.transaction.total.max threshold reached)
Transaction failed and will be retried in 1.0055316591553907s (The allocation of an extra 2.0 MiB would use more than the limit 21.0 GiB. Currently using 21.0 GiB. dbms.memory.transaction.total.max threshold reached)
Querying graph:  61%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▉                                                                  

In [12]:
results = process_results(todo, llm_answers, cypher_results)
with open("../RIW_190_NL_to_KG_benchmarking/results/biomix14_adapted-evaluations.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)

In [13]:
results_df = pd.DataFrame(results)
results_df["has_interaction"] = results_df["count"] > 0
results_df = results_df.merge(questions, left_on="question", right_on="text", how="left")
results_df = results_df.drop(columns=["text"])
results_df['direct'] = ~results_df['question'].str.contains("is not associated")
results_df["answer"] = results_df["direct"] == results_df["has_interaction"]

In [14]:
results_df.to_excel("../RIW_190_NL_to_KG_benchmarking/results/biomix14_adapted-evaluations.xlsx", index=False)
results_df

,model,question,llm_answer,cypher_output,n_cypher_queries,query,success,results,time,count,error,has_interaction,label,direct,answer
0,gpt-4o,Polycythemia Vera is not associated with Gene ...,```cypher\nMATCH (g:Gene)-[:IS_PART_OF]->(asso...,[{'query': 'MATCH (g:Gene)-[:IS_PART_OF]->(ass...,1,MATCH (g:Gene)-[:IS_PART_OF]->(assoc:GeneToDis...,True,"[{'g.approvedSymbol': 'JAK2', 'd.name': 'polyc...",0.479741,10.0,NaN,True,False,False,False
1,gpt-5,Polycythemia Vera is not associated with Gene ...,```cypher\nMATCH (g)\nWHERE any(l IN labels(g)...,[{'query': 'MATCH (g) WHERE any(l IN labels(g)...,1,MATCH (g)\nWHERE any(l IN labels(g) WHERE l IN...,True,"[{'gene_id': 'ensembl:ENSG00000096968', 'gene_...",14.387217,1.0,NaN,True,False,False,False
2,gpt-5.2-2025-12-11,Polycythemia Vera is not associated with Gene ...,```cypher\nMATCH (g:Gene)-[:IS_PART_OF]->(asso...,[{'query': 'MATCH (g:Gene)-[:IS_PART_OF]->(ass...,1,MATCH (g:Gene)-[:IS_PART_OF]->(assoc:GeneToDis...,True,"[{'disease_id': 'efo:0002429', 'disease_name':...",0.174996,25.0,NaN,True,False,False,False
3,claude-sonnet-4-20250514,Polycythemia Vera is not associated with Gene ...,```cypher\nMATCH (g:Gene)-[:IS_PART_OF]->(asso...,[{'query': 'MATCH (g:Gene)-[:IS_PART_OF]->(ass...,1,MATCH (g:Gene)-[:IS_PART_OF]->(assoc:GeneToDis...,True,"[{'g.approvedSymbol': 'JAK2', 'd.name': 'polyc...",0.203502,10.0,NaN,True,False,False,False
4,claude-sonnet-4-5-20250929,Polycythemia Vera is not associated with Gene ...,"Looking at your statement ""Polycythemia Vera i...",[{'query': 'MATCH (g:Gene)-[:IS_PART_OF]->(ass...,1,MATCH (g:Gene)-[:IS_PART_OF]->(assoc:GeneToDis...,True,"[{'Gene': 'JAK2', 'Disease': 'polycythemia ver...",0.244367,10.0,NaN,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,gpt-4o,Smith-Lemli-Opitz Syndrome is associated with ...,```cypher\nMATCH (g:Gene {approvedSymbol: 'TBX...,[{'query': 'MATCH (g:Gene {approvedSymbol: 'TB...,1,MATCH (g:Gene {approvedSymbol: 'TBX5'})-[:IS_P...,True,[],0.158593,0.0,NaN,False,False,True,False
496,gpt-5,Smith-Lemli-Opitz Syndrome is associated with ...,```cypher\nMATCH (g)-[:IS_PART_OF]->(assoc)<-[...,[{'query': 'MATCH (g)-[:IS_PART_OF]->(assoc)<-...,1,MATCH (g)-[:IS_PART_OF]->(assoc)<-[:IS_PART_OF...,True,[],5.667944,0.0,NaN,False,False,True,False
497,gpt-5.2-2025-12-11,Smith-Lemli-Opitz Syndrome is associated with ...,```cypher\nMATCH (g:Gene)-[:IS_PART_OF]->(asso...,[{'query': 'MATCH (g:Gene)-[:IS_PART_OF]->(ass...,1,MATCH (g:Gene)-[:IS_PART_OF]->(assoc:GeneToDis...,True,[],0.103034,0.0,NaN,False,False,True,False
498,claude-sonnet-4-20250514,Smith-Lemli-Opitz Syndrome is associated with ...,```cypher\nMATCH (g:Gene)-[:IS_PART_OF]->(asso...,[{'query': 'MATCH (g:Gene)-[:IS_PART_OF]->(ass...,1,MATCH (g:Gene)-[:IS_PART_OF]->(assoc:GeneToDis...,True,[],0.147883,0.0,NaN,False,False,True,False


In [15]:
# Show raw LLM answers for each model
print("="*60)
print("RAW LLM ANSWERS")
print("="*60)

for idx, row in results_df[0:20].iterrows():
    print(f"\nModel: {row['model']}")
    print(f"Question: {row['question'][:80]}...")
    print(f"LLM Answer:")
    print("-"*60)
    print(row['llm_answer'])
    print("="*60)

RAW LLM ANSWERS

Model: gpt-4o
Question: Polycythemia Vera is not associated with Gene JAK2...
LLM Answer:
------------------------------------------------------------
```cypher
MATCH (g:Gene)-[:IS_PART_OF]->(assoc:GeneToDiseaseAssociation)<-[:IS_PART_OF]-(d:Disease)
WHERE g.approvedSymbol = 'JAK2' AND toLower(d.name) CONTAINS 'polycythemia vera'
RETURN g.approvedSymbol, d.name, assoc.id, assoc.score, assoc.source
LIMIT 10
```


Model: gpt-5
Question: Polycythemia Vera is not associated with Gene JAK2...
LLM Answer:
------------------------------------------------------------
```cypher
MATCH (g)
WHERE any(l IN labels(g) WHERE l IN ['Gene','HumanGene'])
  AND toLower(coalesce(g.approvedSymbol,'')) = 'jak2'
MATCH (d)
WHERE any(l IN labels(d) WHERE l IN ['Disease','Mondo.Disease','Doid.Disease','Efo.Disease','Orphanet.Disease','Hp.Disease','Otar.Disease','Ncit.Disease','Obi.Disease','Ogms.Disease'])
  AND d.name =~ '(?i).*polycyth(a)?emia\s+vera.*'
WITH DISTINCT g, d
OPTIONAL MATCH (g)-[:

In [16]:
# Show cypher outputs for each model
print("="*60)
print("Cypher Outputs")
print("="*60)

for idx, row in results_df[0:6].iterrows():
    print(f"\nModel: {row['model']}")
    print(f"Question: {row['question'][:80]}...")
    print(f"LLM Answer:")
    print("-"*60)
    print(row['cypher_output'])
    print("="*60)

Cypher Outputs

Model: gpt-4o
Question: Polycythemia Vera is not associated with Gene JAK2...
LLM Answer:
------------------------------------------------------------
[{'query': "MATCH (g:Gene)-[:IS_PART_OF]->(assoc:GeneToDiseaseAssociation)<-[:IS_PART_OF]-(d:Disease)\nWHERE g.approvedSymbol = 'JAK2' AND toLower(d.name) CONTAINS 'polycythemia vera'\nRETURN g.approvedSymbol, d.name, assoc.id, assoc.score, assoc.source\nLIMIT 10", 'success': True, 'result': [{'g.approvedSymbol': 'JAK2', 'd.name': 'polycythemia vera', 'assoc.id': '8efaf95fc626cf70f5a9045a2c0a5a84a073301f', 'assoc.score': 0.2, 'assoc.source': 'chembl'}, {'g.approvedSymbol': 'JAK2', 'd.name': 'polycythemia vera', 'assoc.id': '1b684d7ef6a3f270004f06c6b9826d78e1bb14de', 'assoc.score': 0.1, 'assoc.source': 'chembl'}, {'g.approvedSymbol': 'JAK2', 'd.name': 'polycythemia vera', 'assoc.id': 'efbf826210fb759ab3897c614a8e16a54135dd47', 'assoc.score': 1.0, 'assoc.source': 'uniprot_literature'}, {'g.approvedSymbol': 'JAK2', 'd.name':

In [17]:
# Show errors for each model
print("="*60)
print("Error")
print("="*60)

for idx, row in results_df[0:20].iterrows():
    print(f"\nModel: {row['model']}")
    print(f"Question: {row['question'][:80]}...")
    print(f"Error:")
    print("-"*60)
    print(row['error'])
    print("="*60)

Error

Model: gpt-4o
Question: Polycythemia Vera is not associated with Gene JAK2...
Error:
------------------------------------------------------------
nan

Model: gpt-5
Question: Polycythemia Vera is not associated with Gene JAK2...
Error:
------------------------------------------------------------
nan

Model: gpt-5.2-2025-12-11
Question: Polycythemia Vera is not associated with Gene JAK2...
Error:
------------------------------------------------------------
nan

Model: claude-sonnet-4-20250514
Question: Polycythemia Vera is not associated with Gene JAK2...
Error:
------------------------------------------------------------
nan

Model: claude-sonnet-4-5-20250929
Question: Polycythemia Vera is not associated with Gene JAK2...
Error:
------------------------------------------------------------
nan

Model: gpt-4o
Question: Cystic Fibrosis associates Gene CFTR...
Error:
------------------------------------------------------------
nan

Model: gpt-5
Question: Cystic Fibrosis associates Ge

In [18]:
# Calculate the fraction of correct answers for each model
accuracy_df = results_df.groupby('model').apply(lambda x: (x['label'] == x['answer']).mean()).reset_index()
accuracy_df.columns = ['model', 'accuracy']
accuracy_df

/var/folders/xh/h0s_k2yj7vl5lcl4dyjlstxr0000gp/T/ipykernel_85247/2029851061.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  accuracy_df = results_df.groupby('model').apply(lambda x: (x['label'] == x['answer']).mean()).reset_index()


,model,accuracy
0,claude-sonnet-4-20250514,0.90
1,claude-sonnet-4-5-20250929,0.92
2,gpt-4o,0.92
3,gpt-5,0.72
4,gpt-5.2-2025-12-11,0.73
